# Séance 3 — DuckDB & SQL analytique

## Partie 1 : prise en main de DuckDB · Partie 2 : un indice de qualité de l'air pour la France

**Durée indicative : 3h** (Partie 1 ~30-40 min, Partie 2 ~2h-2h20, bonus si le temps le permet).

Dans cette séance, on découvre `duckdb` pour :

- exécuter des requêtes SQL directement sur un `DataFrame` pandas, ou sur des fichiers CSV/Parquet sans les charger en mémoire ;
- faire des agrégations et des jointures entre plusieurs sources ;
- puis, sur un cas plus substantiel (Partie 2), agréger des données hétérogènes provenant de plusieurs capteurs physiques, les nettoyer, et construire un indicateur composite affiché sur une carte interactive.


## Installation des dépendances

Si votre environnement n'est pas encore prêt (voir le `README.md` à la racine), exécutez la cellule suivante.


In [ ]:
%pip install -q -r ../../requirements.txt


---
# Partie 1 — Prise en main de DuckDB

## Rappel : qu'est-ce que DuckDB ?

DuckDB est une base de données **analytique** (adaptée aux agrégations sur de gros volumes, contrairement à une base "transactionnelle" comme PostgreSQL) qui tourne **en process, sans serveur à installer ou à lancer** : on l'importe comme une librairie Python classique. Son gros atout pour nous : il sait lire nativement des `DataFrame` pandas et des fichiers CSV/Parquet, et exécuter du SQL dessus sans étape d'import explicite — DuckDB et pandas sont complémentaires, pas concurrents.


### 1.0. Préparer les données

On réutilise les fichiers générés en séance 1 (`villes_france.csv` et `villes_france_large.parquet`). S'ils n'existent pas encore, la cellule ci-dessous les régénère automatiquement.


In [ ]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

chemin_villes = DATA_DIR / "villes_france.csv"
chemin_large = DATA_DIR / "villes_france_large.parquet"

if not chemin_villes.exists() or not chemin_large.exists():
    print("Fichiers de la séance 1 introuvables : on les régénère rapidement.")

    villes_data = [
        {"ville": "Paris", "region": "Île-de-France", "latitude": 48.8566, "longitude": 2.3522, "population": 2145000},
        {"ville": "Marseille", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.2965, "longitude": 5.3698, "population": 870000},
        {"ville": "Lyon", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7640, "longitude": 4.8357, "population": 522000},
        {"ville": "Toulouse", "region": "Occitanie", "latitude": 43.6047, "longitude": 1.4442, "population": 493000},
        {"ville": "Nice", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.7102, "longitude": 7.2620, "population": 342000},
        {"ville": "Nantes", "region": "Pays de la Loire", "latitude": 47.2184, "longitude": -1.5536, "population": 314000},
        {"ville": "Montpellier", "region": "Occitanie", "latitude": 43.6108, "longitude": 3.8767, "population": 295000},
        {"ville": "Strasbourg", "region": "Grand Est", "latitude": 48.5734, "longitude": 7.7521, "population": 291000},
        {"ville": "Bordeaux", "region": "Nouvelle-Aquitaine", "latitude": 44.8378, "longitude": -0.5792, "population": 260000},
        {"ville": "Lille", "region": "Hauts-de-France", "latitude": 50.6292, "longitude": 3.0573, "population": 233000},
        {"ville": "Rennes", "region": "Bretagne", "latitude": 48.1173, "longitude": -1.6778, "population": 220000},
        {"ville": "Reims", "region": "Grand Est", "latitude": 49.2583, "longitude": 4.0317, "population": 182000},
        {"ville": "Le Havre", "region": "Normandie", "latitude": 49.4944, "longitude": 0.1079, "population": 170000},
        {"ville": "Saint-Étienne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.4397, "longitude": 4.3872, "population": 172000},
        {"ville": "Toulon", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.1242, "longitude": 5.9280, "population": 171000},
        {"ville": "Grenoble", "region": "Auvergne-Rhône-Alpes", "latitude": 45.1885, "longitude": 5.7245, "population": 158000},
        {"ville": "Dijon", "region": "Bourgogne-Franche-Comté", "latitude": 47.3220, "longitude": 5.0415, "population": 156000},
        {"ville": "Angers", "region": "Pays de la Loire", "latitude": 47.4784, "longitude": -0.5632, "population": 152000},
        {"ville": "Nîmes", "region": "Occitanie", "latitude": 43.8367, "longitude": 4.3601, "population": 150000},
        {"ville": "Villeurbanne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7667, "longitude": 4.8794, "population": 149000},
        {"ville": "Clermont-Ferrand", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7772, "longitude": 3.0870, "population": 147000},
        {"ville": "Le Mans", "region": "Pays de la Loire", "latitude": 48.0061, "longitude": 0.1996, "population": 143000},
        {"ville": "Aix-en-Provence", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.5297, "longitude": 5.4474, "population": 143000},
        {"ville": "Brest", "region": "Bretagne", "latitude": 48.3904, "longitude": -4.4861, "population": 139000},
        {"ville": "Tours", "region": "Centre-Val de Loire", "latitude": 47.3941, "longitude": 0.6848, "population": 136000},
        {"ville": "Amiens", "region": "Hauts-de-France", "latitude": 49.8941, "longitude": 2.2957, "population": 133000},
        {"ville": "Limoges", "region": "Nouvelle-Aquitaine", "latitude": 45.8336, "longitude": 1.2611, "population": 131000},
        {"ville": "Annecy", "region": "Auvergne-Rhône-Alpes", "latitude": 45.8992, "longitude": 6.1294, "population": 128000},
        {"ville": "Perpignan", "region": "Occitanie", "latitude": 42.6887, "longitude": 2.8948, "population": 119000},
        {"ville": "Metz", "region": "Grand Est", "latitude": 49.1193, "longitude": 6.1757, "population": 116000},
    ]
    pd.DataFrame(villes_data).to_csv(chemin_villes, index=False)

    rng = np.random.default_rng(42)
    n = 5000
    df_large = pd.DataFrame({
        "ville": [f"Ville_{i:05d}" for i in range(n)],
        "latitude": rng.uniform(41.0, 51.5, n),
        "longitude": rng.uniform(-5.0, 9.5, n),
        "population": rng.integers(500, 200_000, n),
    })
    df_large.to_parquet(chemin_large, index=False)
else:
    print("Fichiers de la séance 1 trouvés, on les réutilise.")

df_villes = pd.read_csv(chemin_villes)
df_villes.head()


### 1.1. Une requête SQL sur un DataFrame pandas

`duckdb.sql(...)` peut référencer directement une variable Python contenant un `DataFrame` — pas besoin de l'importer explicitement dans une base. Le résultat est un objet DuckDB qu'on reconvertit en `DataFrame` pandas avec `.df()`.


In [ ]:
resultat = duckdb.sql("""
    SELECT ville, region, population
    FROM df_villes
    WHERE population > 200000
    ORDER BY population DESC
""").df()

resultat


### 1.2. Une requête SQL directement sur un fichier Parquet

On peut interroger un fichier Parquet **sur disque** sans jamais le charger entièrement en mémoire avec pandas : DuckDB lit le fichier lui-même, via `read_parquet(...)`.


In [ ]:
resultat = duckdb.sql(f"""
    SELECT COUNT(*) AS nb_villes, AVG(population) AS population_moyenne
    FROM read_parquet('{chemin_large}')
""").df()

resultat


### 1.3. Agrégations et jointure entre deux fichiers

On génère un second jeu de données synthétique — une météo moyenne par ville — pour illustrer une jointure SQL entre deux fichiers.


In [ ]:
rng = np.random.default_rng(7)

df_meteo = pd.DataFrame({
    "ville": df_villes["ville"],
    "temperature_moyenne_c": (20 - (df_villes["latitude"] - 43) * 0.5 + rng.normal(0, 0.8, len(df_villes))).round(1),
    "precipitation_mm_an": rng.integers(550, 1200, len(df_villes)),
})

chemin_meteo = DATA_DIR / "meteo_villes.csv"
df_meteo.to_csv(chemin_meteo, index=False)
df_meteo.head()


In [ ]:
resultat = duckdb.sql(f"""
    SELECT v.region,
           COUNT(*) AS nb_villes,
           AVG(v.population) AS population_moyenne,
           AVG(m.temperature_moyenne_c) AS temperature_moyenne
    FROM read_csv_auto('{chemin_villes}') v
    JOIN read_csv_auto('{chemin_meteo}') m ON v.ville = m.ville
    GROUP BY v.region
    ORDER BY population_moyenne DESC
""").df()

resultat


### 1.4. pandas vs DuckDB : un temps d'exécution

On génère un fichier Parquet **d'un million de lignes** pour comparer, sur une même agrégation (`GROUP BY` + `AVG`), pandas et DuckDB. Gardez ce résultat en tête : en Partie 2, on refera des agrégations sur plusieurs millions de lignes (produites par un `CROSS JOIN`), et ce sera précisément ce qui rend l'approche praticable.


In [ ]:
rng = np.random.default_rng(0)
n_perf = 1_000_000
regions = ["IDF", "ARA", "PACA", "Occitanie", "Nouvelle-Aquitaine", "Bretagne", "Grand Est", "Hauts-de-France", "Normandie", "Pays de la Loire"]

df_perf = pd.DataFrame({
    "region": rng.choice(regions, n_perf),
    "population": rng.integers(500, 200_000, n_perf),
})

chemin_perf = DATA_DIR / "villes_perf.parquet"
df_perf.to_parquet(chemin_perf, index=False)
print(f"Fichier de {n_perf:,} lignes généré : {chemin_perf}")


In [ ]:
debut = time.perf_counter()
df_charge = pd.read_parquet(chemin_perf)
resultat_pandas = df_charge.groupby("region")["population"].mean()
duree_pandas = time.perf_counter() - debut

debut = time.perf_counter()
resultat_duckdb = duckdb.sql(f"""
    SELECT region, AVG(population) AS population_moyenne
    FROM read_parquet('{chemin_perf}')
    GROUP BY region
""").df()
duree_duckdb = time.perf_counter() - debut

print(f"pandas (read_parquet + groupby) : {duree_pandas * 1000:.1f} ms")
print(f"duckdb (lecture + agrégation)   : {duree_duckdb * 1000:.1f} ms")


### 🧪 À vous de jouer : 3 requêtes SQL à trous

Complétez les trois requêtes ci-dessous (chaque `<TODO>` doit être remplacé par un mot-clé SQL, un nom de colonne ou une valeur).


**1) Les 5 villes les plus peuplées** (sur `df_villes`) :


In [ ]:
resultat_a = duckdb.sql("""
    <TODO>
""").df()

resultat_a


**2) Nombre de villes et population moyenne par région, triés par nombre de villes décroissant** :


In [ ]:
resultat_b = duckdb.sql("""
    <TODO>
""").df()

resultat_b


**3) Villes où la température moyenne dépasse 14°C** (jointure entre `villes_france.csv` et `meteo_villes.csv`) :


In [ ]:
resultat_c = duckdb.sql(f"""
    <TODO>
""").df()

resultat_c

---
# Partie 2 — Projet : un indice de qualité de l'air pour la France

## Contexte

Une base de données `data/qualite_air.duckdb` contient les relevés de **quatre réseaux de capteurs** atmosphériques répartis sur la France métropolitaine (météo, qualité de l'air urbain, ozone rural, capteurs citoyens — voir `generation_base_donnees.ipynb` pour le détail de chaque réseau). Cette base a les défauts d'une vraie base de données : un identifiant de capteur qui ne s'appelle pas pareil d'une table à l'autre, des colonnes sans intérêt, des valeurs manquantes et aberrantes.

## Objectif

Construire un **indice de qualité de l'air à 5 niveaux** (Très bon, Bon, Moyen, Dégradé, Très dégradé) sur un maillage régulier de 10×10 km de la France métropolitaine, et l'afficher sur une carte interactive.

## Compétences mobilisées

- **SQL** : `ALTER TABLE ... RENAME COLUMN`, agrégations conditionnelles, `WHERE` multi-critères, `UNION ALL`, `CROSS JOIN` pondéré, `GROUP BY`.
- **Nettoyage de données hétérogènes** : harmoniser des schémas, repérer et traiter valeurs manquantes/aberrantes, filtrer par critère métier.
- **Modélisation** : construire un indicateur composite à partir de règles réalistes (sous-indices, règle du "pire polluant", modulation).
- **Géomatique** : reprojection de CRS (rappel séance 1), maillage régulier, interpolation spatiale par noyau gaussien.
- **Restitution** : carte interactive (folium), visualisation combinée capteurs + champ interpolé (plotly).

## Étapes

1. Explorer les 4 tables et repérer l'hétérogénéité des identifiants.
2. Harmoniser les colonnes d'identifiant (`id_capteur` partout).
3. Nettoyer (valeurs manquantes, aberrantes) et filtrer (critère métier).
4. Regrouper les capteurs de PM10 des deux réseaux qui le mesurent.
5. Construire un maillage régulier 10×10 km de la France métropolitaine.
6. Interpoler chaque variable sur le maillage par pondération gaussienne (SQL).
7. Construire la fonction de qualité de l'air et l'appliquer au maillage.
8. Écrire le résultat dans une nouvelle table `air_quality_index`, afficher la carte interactive.
9. **Bonus** : cartes d'interpolation par variable avec capteurs au survol, indice de confiance, seuil "donnée non disponible".


## (Optionnel mais recommandé) Installer un outil pour explorer visuellement la base de données

Tout le TP se fait en interrogeant `data/qualite_air.duckdb` depuis Python avec la librairie `duckdb` — ça reste le point obligatoire. Mais pouvoir aussi *regarder* les tables dans une interface, en dehors du notebook, aide à se faire une idée concrète de ce qu'on manipule. On installe pour ça le binaire officiel **DuckDB CLI** : gratuit, identique sur Mac et Windows, et surtout c'est le même moteur que celui utilisé en Python — pas de format intermédiaire, pas de risque d'incohérence.

**macOS**
1. Ouvrez un terminal.
2. Installez le binaire :
   - avec [Homebrew](https://brew.sh) : `brew install duckdb`
   - sans Homebrew : `curl https://install.duckdb.org | sh`
3. Vérifiez : `duckdb --version`.

**Windows**
1. Rendez-vous sur la page d'installation officielle (duckdb.org → *Installation* → *CLI* → *Windows*) et téléchargez l'archive `duckdb_cli-windows-amd64.zip`.
2. Extrayez `duckdb.exe` dans un dossier simple (par ex. `C:\duckdb\`).
3. Ouvrez PowerShell dans ce dossier (clic droit → *Ouvrir dans le terminal*) et vérifiez : `.\duckdb.exe --version`.
   *(Astuce : ajoutez ce dossier à la variable d'environnement `PATH` pour pouvoir taper simplement `duckdb` depuis n'importe quel dossier.)*


**Lancer l'interface.** Une fois `generation_base_donnees.ipynb` exécuté (ou si `data/qualite_air.duckdb` est déjà présent), ouvrez un terminal dans le dossier `exercices/` et lancez :

```
duckdb -ui data/qualite_air.duckdb
```

Un onglet s'ouvre dans votre navigateur avec une interface pour parcourir les tables, voir leur schéma et exécuter du SQL — exactement la même base que celle interrogée depuis Python dans ce notebook. (Si la commande n'est pas reconnue, votre version de DuckDB est probablement trop ancienne pour l'extension UI intégrée ; une simple réinstallation avec la procédure ci-dessus résout le problème dans la grande majorité des cas.)

**Ce que vous devriez y trouver.** Au départ, 4 tables : `releves_meteo`, `releves_qualite_air_urbain`, `releves_ozone_rural`, `releves_citoyen_pm` — chacune avec sa propre colonne d'identifiant (nommée différemment selon la table), quelques colonnes sans intérêt pour l'indice de qualité de l'air, et des valeurs manquantes ou aberrantes. Au fil du TP, de nouvelles tables apparaîtront (les versions nettoyées `..._propre`, les tables `capteurs_*`, `maille`, et enfin `air_quality_index`) — rechargez la page de l'interface pour les voir.

*Vous avez déjà un autre outil (DBeaver, TablePlus...) qui sait ouvrir un fichier DuckDB ? Il convient tout aussi bien : cette étape est un complément visuel facultatif, pas une dépendance du TP.*


## 1. Connexion et exploration des schémas

On se connecte au fichier généré par `generation_base_donnees.ipynb`. Si le fichier n'existe pas, il faut d'abord exécuter ce notebook.


In [ ]:
import duckdb

CHEMIN_DB = DATA_DIR / "qualite_air.duckdb"
if not CHEMIN_DB.exists():
    raise FileNotFoundError("Lancez d'abord generation_base_donnees.ipynb pour créer la base de données.")

con = duckdb.connect(str(CHEMIN_DB))
con.sql("SHOW TABLES")


In [ ]:
for table in ["releves_meteo", "releves_qualite_air_urbain", "releves_ozone_rural", "releves_citoyen_pm"]:
    print(f"--- {table} ---")
    print(con.sql(f"DESCRIBE {table}"))


On retrouve bien 4 noms différents pour la colonne qui identifie un capteur de façon unique : `id_station_meteo`, `code_capteur_aq`, `ref_station_o3`, `capteur_uid_pm`. Avant de pouvoir regrouper ces tables, il faut les harmoniser.


## 2. Harmoniser les identifiants

`ALTER TABLE ... RENAME COLUMN ... TO ...` renomme une colonne en place, sans recréer la table.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
RENOMMAGES = {
    "releves_meteo": "<TODO>",
    "releves_qualite_air_urbain": "<TODO>",
    "releves_ozone_rural": "<TODO>",
    "releves_citoyen_pm": "<TODO>",
}

for table, ancien_nom in RENOMMAGES.items():
    colonnes = [r[0] for r in con.sql(f"DESCRIBE {table}").fetchall()]
    if "id_capteur" not in colonnes:
        con.execute(f"ALTER TABLE {table} RENAME COLUMN {ancien_nom} TO id_capteur")

con.sql("DESCRIBE releves_meteo")


## 3. Nettoyage : valeurs manquantes, valeurs aberrantes, filtrage métier

**Valeurs manquantes.** On compte les `NULL` par colonne avec une agrégation conditionnelle (`SUM(CASE WHEN ... IS NULL THEN 1 ELSE 0 END)`) — l'équivalent SQL de `isna().sum()`.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
con.sql("""
    SELECT
        SUM(CASE WHEN <TODO> IS NULL THEN 1 ELSE 0 END) AS nan_temperature,
        SUM(CASE WHEN <TODO> IS NULL THEN 1 ELSE 0 END) AS nan_humidite,
        SUM(CASE WHEN <TODO> IS NULL THEN 1 ELSE 0 END) AS nan_precipitation,
        SUM(CASE WHEN <TODO> IS NULL THEN 1 ELSE 0 END) AS nan_vent,
        COUNT(*) AS total
    FROM releves_meteo
""")


**Valeurs aberrantes et filtrage métier.** On fixe des bornes physiquement réalistes par variable (cohérentes avec les ordres de grandeur donnés dans `generation_base_donnees.ipynb`) et on filtre avec `WHERE ... BETWEEN ...`. Une comparaison avec `NULL` ne vaut jamais vrai en SQL : un simple `WHERE colonne BETWEEN a AND b` élimine donc *aussi* les lignes où `colonne` est `NULL`, sans avoir besoin d'un `dropna()` séparé.

On ajoute en même temps un filtre métier sur `releves_qualite_air_urbain` : on ne garde que les stations de type `'fond urbain'` (on écarte `'trafic'` et `'industriel'`, des implantations qui mesurent une pollution très locale non représentative de la qualité de l'air ambiante — c'est d'ailleurs la règle suivie par les vrais indices officiels).


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE releves_meteo_propre AS
    SELECT * FROM releves_meteo
    WHERE temperature_c BETWEEN <TODO> AND <TODO>
      AND humidite_pct BETWEEN <TODO> AND <TODO>
      AND precipitation_mm_jour BETWEEN <TODO> AND <TODO>
      AND vent_vitesse_kmh BETWEEN <TODO> AND <TODO>
""")

con.execute("""
    CREATE OR REPLACE TABLE releves_qualite_air_urbain_propre AS
    SELECT * FROM releves_qualite_air_urbain
    WHERE no2_ugm3 BETWEEN <TODO> AND <TODO>
      AND pm10_ugm3 BETWEEN <TODO> AND <TODO>
      AND pm25_ugm3 BETWEEN <TODO> AND <TODO>
      AND type_implantation = <TODO>
""")

con.execute("""
    CREATE OR REPLACE TABLE releves_ozone_rural_propre AS
    SELECT * FROM releves_ozone_rural
    WHERE o3_ugm3 BETWEEN <TODO> AND <TODO>
""")

con.execute("""
    CREATE OR REPLACE TABLE releves_citoyen_pm_propre AS
    SELECT * FROM releves_citoyen_pm
    WHERE pm10_ugm3 BETWEEN <TODO> AND <TODO>
""")

for table in ["releves_meteo", "releves_meteo_propre", "releves_qualite_air_urbain", "releves_qualite_air_urbain_propre",
              "releves_ozone_rural", "releves_ozone_rural_propre", "releves_citoyen_pm", "releves_citoyen_pm_propre"]:
    print(table, "->", con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0], "lignes")


## 4. Regrouper les capteurs par variable (dont l'union des deux réseaux PM10)

Pour chaque variable utile à l'indice (NO2, PM10, PM2.5, O3, vent), on construit une table de capteurs au schéma identique (`id_capteur`, `latitude`, `longitude`, `valeur`, et leur position en Lambert-93 `x_l93`/`y_l93` pour calculer des distances en mètres — même logique de reprojection que la séance 1). PM10 est mesuré par **deux réseaux différents** (`releves_qualite_air_urbain_propre` et `releves_citoyen_pm_propre`) : on les combine avec `UNION ALL` pour disposer d'un pool unique de capteurs PM10, officiels et citoyens mélangés.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
from pyproj import Transformer

VERS_L93 = Transformer.from_crs(4326, 2154, always_xy=True)
VERS_WGS84 = Transformer.from_crs(2154, 4326, always_xy=True)


def creer_table_capteurs(nom_table, requete_sql):
    """Exécute requete_sql (qui doit renvoyer id_capteur, latitude, longitude, valeur), ajoute la
    projection Lambert-93, et écrit le résultat dans une nouvelle table DuckDB."""
    df = con.sql(requete_sql).df()
    df["x_l93"], df["y_l93"] = VERS_L93.transform(df["longitude"].to_numpy(), df["latitude"].to_numpy())
    con.register("df_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {nom_table} AS SELECT * FROM df_tmp")
    con.unregister("df_tmp")
    return df


creer_table_capteurs("capteurs_no2", """
    SELECT id_capteur, latitude, longitude, <TODO> AS valeur
    FROM releves_qualite_air_urbain_propre WHERE <TODO> IS NOT NULL
""")

creer_table_capteurs("capteurs_pm25", """
    SELECT id_capteur, latitude, longitude, <TODO> AS valeur
    FROM releves_qualite_air_urbain_propre WHERE <TODO> IS NOT NULL
""")

creer_table_capteurs("capteurs_o3", """
    SELECT id_capteur, latitude, longitude, <TODO> AS valeur
    FROM <TODO> WHERE o3_ugm3 IS NOT NULL
""")

creer_table_capteurs("capteurs_vent", """
    SELECT id_capteur, latitude, longitude, <TODO> AS valeur
    FROM releves_meteo_propre WHERE vent_vitesse_kmh IS NOT NULL
""")

creer_table_capteurs("capteurs_pm10", """
    SELECT id_capteur, latitude, longitude, pm10_ugm3 AS valeur
    FROM releves_qualite_air_urbain_propre WHERE pm10_ugm3 IS NOT NULL
    <TODO>
    SELECT id_capteur, latitude, longitude, pm10_ugm3 AS valeur
    FROM releves_citoyen_pm_propre WHERE pm10_ugm3 IS NOT NULL
""")

for table in ["capteurs_no2", "capteurs_pm25", "capteurs_o3", "capteurs_vent", "capteurs_pm10"]:
    print(table, "->", con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0], "capteurs")


## 5. Un maillage régulier 10×10 km de la France métropolitaine

On reprend le contour simplifié du notebook de génération, et on le reprojette en Lambert-93 (mètres) pour construire une grille **régulière** de carrés de 10 km de côté — en degrés, les mailles ne seraient pas réellement carrées (la distance que représente 1° de longitude varie avec la latitude). On ne garde que les mailles dont le centre tombe à l'intérieur du contour.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, box
from shapely.ops import transform as shp_transform

FRANCE_METRO_POLYGON = Polygon([
    (2.37, 51.03), (3.28, 50.71), (4.00, 50.28), (4.72, 49.93), (5.78, 49.53),
    (6.90, 49.18), (7.77, 48.58), (7.45, 47.55), (6.35, 46.90), (6.87, 46.02),
    (6.68, 45.20), (6.65, 44.90), (7.56, 44.09), (7.50, 43.78), (7.27, 43.70),
    (6.65, 43.40), (5.93, 43.12), (5.37, 43.30), (3.93, 43.53), (3.03, 42.70),
    (2.87, 42.47), (1.44, 42.55), (-0.44, 42.78), (-1.77, 43.37), (-1.56, 43.48),
    (-1.25, 44.65), (-1.03, 45.63), (-1.15, 46.16), (-2.21, 47.27), (-3.13, 47.48),
    (-3.37, 47.75), (-4.78, 48.38), (-3.98, 48.72), (-1.99, 48.65), (-1.62, 49.65),
    (0.11, 49.49), (1.08, 49.93), (1.86, 50.95), (2.37, 51.03),
])  # même contour simplifié que generation_base_donnees.ipynb

polygone_l93 = shp_transform(lambda lon, lat: VERS_L93.transform(lon, lat), FRANCE_METRO_POLYGON)
minx, miny, maxx, maxy = polygone_l93.bounds
PAS_MAILLE_M = <TODO>  # en mètres, pour un maillage de 10 km de côté

xs = np.arange(minx, maxx, PAS_MAILLE_M)
ys = np.arange(miny, maxy, PAS_MAILLE_M)
xx, yy = np.meshgrid(xs, ys)
x_centres, y_centres = xx.ravel(), yy.ravel()

gdf_candidats = gpd.GeoDataFrame(
    {"x_l93": x_centres, "y_l93": y_centres},
    geometry=gpd.points_from_xy(x_centres, y_centres),
    crs="EPSG:2154",
)
gdf_maille = gdf_candidats[gdf_candidats.within(polygone_l93)].reset_index(drop=True)
gdf_maille["id_maille"] = gdf_maille.index

demi_pas = PAS_MAILLE_M / 2
gdf_maille["geometry"] = gdf_maille.apply(
    lambda r: box(r["x_l93"] - demi_pas, r["y_l93"] - demi_pas, r["x_l93"] + demi_pas, r["y_l93"] + demi_pas), axis=1
)
gdf_maille_wgs84 = gdf_maille.set_geometry(gdf_maille.geometry, crs="EPSG:2154").to_crs(4326)

lon_centres, lat_centres = VERS_WGS84.transform(gdf_maille["x_l93"].to_numpy(), gdf_maille["y_l93"].to_numpy())
gdf_maille["longitude"], gdf_maille["latitude"] = lon_centres, lat_centres

df_maille = gdf_maille[["id_maille", "x_l93", "y_l93", "latitude", "longitude"]].copy()
con.register("df_maille_tmp", df_maille)
con.execute("CREATE OR REPLACE TABLE maille AS SELECT * FROM df_maille_tmp")
con.unregister("df_maille_tmp")

print(f"{len(df_maille)} mailles de {PAS_MAILLE_M / 1000:.0f} km de côté générées.")


## 6. Interpoler chaque variable sur le maillage (SQL)

On applique la formule donnée en introduction de séance : pour une maille `y` et une variable `j`,

$$f_j(y) = \frac{\sum_i w_i \cdot f_j(x_i)}{\sum_i w_i} \quad\text{avec}\quad w_i = \exp\left(-\left(\frac{D(y, x_i)}{l_j}\right)^2\right)$$

où `D(y, x_i)` est la distance (en mètres, calculée directement à partir des coordonnées Lambert-93) entre la maille et le capteur `i`, et `l_j` une longueur caractéristique choisie par variable. Cette expression se traduit directement en SQL par un `CROSS JOIN` entre `maille` et la table de capteurs, suivi d'un `GROUP BY` sur la maille — DuckDB encaisse sans peine le produit cartésien (quelques milliers de mailles × quelques centaines de capteurs = plusieurs millions de lignes), exactement comme la comparaison de performance de la Partie 1 le suggérait.

`Σw_i` (le dénominateur) est conservé à part : c'est une mesure de **combien de capteurs proches** ont contribué à l'estimation, et servira de base à l'indice de confiance en bonus.

Le choix de `l_j` est un choix de modélisation : plus il est petit, plus l'estimation colle aux capteurs les plus proches (quitte à créer des "bulles" autour de chaque capteur) ; plus il est grand, plus elle est lissée (quitte à gommer les variations locales réelles). Les ordres de grandeur ci-dessous reflètent la densité de chaque réseau (NO2/PM, denses en ville, varient vite dans l'espace → `l` petit ; vent, mesuré par un réseau clairsemé et spatialement lisse, varie lentement → `l` grand).


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
LONGUEURS_CARACTERISTIQUES_KM = {
    "no2": <TODO>,
    "pm10": <TODO>,
    "pm25": <TODO>,
    "o3": <TODO>,
    "vent": <TODO>,
}


def requete_interpolation(nom_variable, longueur_km):
    longueur_m = longueur_km * 1000
    return f"""
        WITH poids AS (
            SELECT
                m.id_maille,
                c.valeur,
                <TODO> AS w
            FROM maille m
            CROSS JOIN capteurs_{nom_variable} c
        )
        SELECT
            id_maille,
            SUM(w * valeur) / SUM(w) AS estimation,
            SUM(w) AS confiance
        FROM poids
        GROUP BY <TODO>
        ORDER BY id_maille
    """


for variable, longueur_km in LONGUEURS_CARACTERISTIQUES_KM.items():
    resultat = con.sql(requete_interpolation(variable, longueur_km)).df()
    resultat = resultat.rename(columns={"estimation": f"{variable}_estime", "confiance": f"confiance_{variable}"})
    df_maille = df_maille.merge(resultat, on="id_maille", how="left")

df_maille.describe()


## 7. Construire la fonction de qualité de l'air

On applique la méthodologie décrite dans `generation_base_donnees.ipynb` : un sous-indice (1 à 5) par polluant via des seuils de concentration, l'indice global est le **pire** des sous-indices, puis une **modulation météo** selon le vent.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
import pandas as pd

SEUILS_SOUS_INDICE = {
    "pm10_estime": [20, 40, 50, 80],
    "pm25_estime": [10, 20, 25, 50],
    "no2_estime": [40, 90, 120, 230],
    "o3_estime": [50, 100, 130, 180],
}


def niveau_sous_indice(valeur, seuils):
    """Renvoie 1 à 5 selon la première borne de `seuils` (liste croissante de 4 valeurs) que `valeur`
    ne dépasse pas, ou 5 si elle dépasse toutes les bornes. Renvoie NaN si `valeur` est manquante."""
    # <TODO> : à écrire.


for colonne, seuils in SEUILS_SOUS_INDICE.items():
    df_maille[f"niveau_{colonne}"] = df_maille[colonne].apply(lambda v: niveau_sous_indice(v, seuils))

colonnes_niveaux = [f"niveau_{c}" for c in SEUILS_SOUS_INDICE]
df_maille["niveau_pire"] = <TODO>  # le pire (= le plus grand) des 4 sous-indices, ligne par ligne


def moduler_avec_le_vent(niveau, vitesse_vent_kmh):
    """Dégrade le niveau d'un cran si le vent est < 5 km/h, l'améliore d'un cran si le vent est
    > 25 km/h (sans sortir de l'intervalle [1, 5]). Ne change rien si le niveau ou le vent est NaN."""
    # <TODO> : à écrire.


df_maille["niveau_final"] = df_maille.apply(lambda r: moduler_avec_le_vent(r["niveau_pire"], r["vent_estime"]), axis=1)

LIBELLES_NIVEAU = {1: "Très bon", 2: "Bon", 3: "Moyen", 4: "Dégradé", 5: "Très dégradé"}
df_maille["qualite_air"] = df_maille["niveau_final"].map(LIBELLES_NIVEAU)

df_maille["qualite_air"].value_counts()


## 8. Écrire le résultat en base et afficher la carte

On écrit `df_maille` dans une nouvelle table `air_quality_index` de la même base `qualite_air.duckdb`, puis on affiche le résultat avec `folium`. Les mailles sont reconstruites comme de vrais carrés (en Lambert-93, via `geopandas`) plutôt qu'approximées par des cercles.


In [ ]:
con.execute("CREATE OR REPLACE TABLE air_quality_index AS SELECT * FROM df_maille")
con.sql("SELECT COUNT(*) FROM air_quality_index")


In [ ]:
import folium

COULEURS_NIVEAU = {1: "#2ecc71", 2: "#a3d900", 3: "#f1c40f", 4: "#e67e22", 5: "#e74c3c"}

gdf_carte = gdf_maille_wgs84.copy()
gdf_carte["niveau_final"] = df_maille["niveau_final"].values
gdf_carte["qualite_air"] = df_maille["qualite_air"].values
gdf_carte = gdf_carte.dropna(subset=["niveau_final"])

carte = folium.Map(location=[46.6, 2.5], zoom_start=6, tiles="cartodbpositron")
folium.GeoJson(
    gdf_carte,
    style_function=lambda f: {
        "fillColor": COULEURS_NIVEAU[int(f["properties"]["niveau_final"])],
        "color": None,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(fields=["qualite_air"], aliases=["Qualité de l'air"]),
).add_to(carte)
carte


**Version plotly.** La carte folium ci-dessus est riche (zoom, tuiles réelles) mais s'ouvre mal inline dans certains environnements de notebook. Voici une version `plotly` équivalente, qui s'affiche directement dans la sortie de la cellule — avec un vrai fond de carte (littoral, frontières) plutôt que notre contour simplifié, obtenu via `scope="europe"` recentré sur la France (plotly n'a pas de `scope` au niveau pays, seulement continent).


In [ ]:
import plotly.express as px

CONFIG_GEO_FRANCE = dict(
    scope="europe",
    center=dict(lon=2.5, lat=46.6),
    projection_scale=6,
    showland=True, landcolor="whitesmoke",
    showcountries=True, countrycolor="lightgray",
)

df_carte = df_maille.dropna(subset=["niveau_final"])
fig = px.scatter_geo(
    df_carte, lat="latitude", lon="longitude", color="qualite_air",
    category_orders={"qualite_air": ["Très bon", "Bon", "Moyen", "Dégradé", "Très dégradé"]},
    color_discrete_map={"Très bon": "#2ecc71", "Bon": "#a3d900", "Moyen": "#f1c40f", "Dégradé": "#e67e22", "Très dégradé": "#e74c3c"},
    hover_data={"id_maille": True, "no2_estime": ":.1f", "pm10_estime": ":.1f"},
    title="Indice de qualité de l'air — France métropolitaine",
)
fig.update_traces(marker=dict(size=6, symbol="square"))
fig.update_geos(**CONFIG_GEO_FRANCE)
fig.update_layout(height=650)
fig.show()


---
## Bonus 1 — Interpolation par variable, avec les capteurs au survol

On combine deux traces Plotly : le champ interpolé sur tout le maillage (petits points colorés, quasi continus à l'échelle de la carte) et les capteurs bruts par-dessus (avec leur identifiant affiché au survol de la souris — pratique pour repérer un capteur suspect). Exemple sur le NO2 ; le même code fonctionne pour PM10, PM2.5, O3 ou le vent en changeant simplement les noms de colonnes.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
import plotly.graph_objects as go

df_capteurs_no2 = con.sql("SELECT id_capteur, latitude, longitude, valeur FROM capteurs_no2").df()

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=df_maille["longitude"], lat=df_maille["latitude"],
    mode="markers",
    marker=dict(size=4, color=df_maille["no2_estime"], colorscale="YlOrRd", opacity=0.6,
                colorbar=dict(title="NO2 (µg/m³)")),
    name="champ interpolé", hoverinfo="skip",
))
fig.add_trace(go.Scattergeo(
    lon=df_capteurs_no2["longitude"], lat=df_capteurs_no2["latitude"],
    mode="markers",
    marker=dict(size=6, color="black", symbol="circle-open"),
    name="capteurs",
    text=<TODO>,  # une chaîne par capteur combinant id_capteur et valeur, pour l'affichage au survol
    hoverinfo="text",
))
fig.update_geos(scope="europe", center=dict(lon=2.5, lat=46.6), projection_scale=6, showland=True, landcolor="whitesmoke")
fig.update_layout(title="NO2 — interpolation et capteurs sources", height=600)
fig.show()


## Bonus 2 — Indice de confiance

Le dénominateur `Σw_i` calculé à l'étape 6 est une mesure directe de densité de capteurs pondérée par la distance : grand quand `y` est entouré de capteurs proches, proche de zéro quand `y` est loin de tout. On en déduit un indice de confiance **global** en prenant le **minimum** sur les 4 polluants (comme pour l'indice lui-même, c'est le maillon le plus faible qui compte : inutile d'avoir une estimation de PM10 très fiable si l'estimation d'ozone au même endroit est construite à partir d'un capteur à 150 km).


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
df_maille["confiance"] = df_maille[<TODO>].min(axis=1)  # les 4 colonnes confiance_* des polluants
df_maille["confiance"].describe()


La distribution est très étalée (quelques mailles très proches d'un capteur, beaucoup de mailles rurales loin de tout). On regarde les quantiles pour choisir un seuil de décision plutôt que d'en inventer un arbitrairement : ici, on choisit de considérer qu'en dessous du **premier quartile**, l'estimation n'est pas assez étayée pour être affichée.


🧪 **À vous de jouer** — complétez les `<TODO>`.


In [ ]:
print(df_maille["confiance"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

SEUIL_CONFIANCE = df_maille["confiance"].quantile(<TODO>)  # à vous de choisir, en regardant la distribution ci-dessus
df_maille["qualite_air_fiable"] = np.where(
    df_maille["confiance"] >= SEUIL_CONFIANCE, df_maille["qualite_air"], <TODO>
)

print(f"Seuil retenu : {SEUIL_CONFIANCE:.2e}")
df_maille["qualite_air_fiable"].value_counts()


On met à jour la table en base pour inclure l'indice de confiance et sa version seuillée, puis on recalcule la carte en y intégrant les deux informations (couleur = qualité de l'air si fiable, gris si non disponible ; confiance affichée au survol).


In [ ]:
con.execute("CREATE OR REPLACE TABLE air_quality_index AS SELECT * FROM df_maille")

COULEURS_NIVEAU_BONUS = {1: "#2ecc71", 2: "#a3d900", 3: "#f1c40f", 4: "#e67e22", 5: "#e74c3c"}
GRIS_NON_DISPONIBLE = "#bdc3c7"

gdf_carte_bonus = gdf_maille_wgs84.copy()
gdf_carte_bonus["niveau_final"] = df_maille["niveau_final"].values
gdf_carte_bonus["qualite_air_fiable"] = df_maille["qualite_air_fiable"].values
gdf_carte_bonus["confiance"] = df_maille["confiance"].round(4).values
gdf_carte_bonus = gdf_carte_bonus.dropna(subset=["niveau_final"])


def couleur_maille(feature):
    if feature["properties"]["qualite_air_fiable"] == "Donnée non disponible":
        return GRIS_NON_DISPONIBLE
    return COULEURS_NIVEAU_BONUS[int(feature["properties"]["niveau_final"])]


carte_bonus = folium.Map(location=[46.6, 2.5], zoom_start=6, tiles="cartodbpositron")
folium.GeoJson(
    gdf_carte_bonus,
    style_function=lambda f: {"fillColor": couleur_maille(f), "color": None, "fillOpacity": 0.7},
    tooltip=folium.GeoJsonTooltip(fields=["qualite_air_fiable", "confiance"], aliases=["Qualité de l'air", "Confiance"]),
).add_to(carte_bonus)
carte_bonus


**Version plotly** de la même carte, avec le niveau de confiance affiché au survol.


In [ ]:
fig = px.scatter_geo(
    df_maille.dropna(subset=["niveau_final"]), lat="latitude", lon="longitude", color="qualite_air_fiable",
    category_orders={"qualite_air_fiable": ["Très bon", "Bon", "Moyen", "Dégradé", "Très dégradé", "Donnée non disponible"]},
    color_discrete_map={
        "Très bon": "#2ecc71", "Bon": "#a3d900", "Moyen": "#f1c40f", "Dégradé": "#e67e22",
        "Très dégradé": "#e74c3c", "Donnée non disponible": "#bdc3c7",
    },
    hover_data={"confiance": ":.2e"},
    title="Indice de qualité de l'air, avec seuil de confiance",
)
fig.update_traces(marker=dict(size=6, symbol="square"))
fig.update_geos(**CONFIG_GEO_FRANCE)
fig.update_layout(height=650)
fig.show()


## Ce qu'on a vu aujourd'hui

- DuckDB exécute du SQL directement sur des `DataFrame` pandas ou des fichiers CSV/Parquet, avec de bonnes performances même sur un `CROSS JOIN` de plusieurs millions de lignes.
- Fusionner des sources hétérogènes commence souvent par renommer des colonnes (`ALTER TABLE ... RENAME COLUMN`) avant même de pouvoir les joindre ou les unioner.
- Un `WHERE ... BETWEEN` élimine implicitement les `NULL`, et un filtre métier (ici `type_implantation`) peut être aussi important qu'un filtre statistique pour la qualité d'un résultat.
- Une interpolation spatiale par noyau gaussien se traduit directement en SQL par un `CROSS JOIN` + `GROUP BY` pondéré, et le dénominateur de la moyenne pondérée est en lui-même une mesure de confiance.
- Un indicateur composite réaliste (qualité de l'air) combine des règles simples (seuils, pire sous-indice, modulation) plutôt qu'une moyenne brute de tout ce qui est mesurable.

**La semaine prochaine (séance 4)** : on va chercher de vraies données spatiales sur les catalogues Copernicus et NASA Earthdata, avec de premiers identifiants API.
